### Célula 1 — Verificando o histórico de versões da Silver

In [0]:
from delta.tables import DeltaTable                                  # biblioteca Delta Lake

SILVER_PATH = "/Volumes/workspace/default/raw/silver/"              # caminho da Silver

# Acessando a tabela Delta
delta_silver = DeltaTable.forPath(spark, SILVER_PATH)               # instancia a tabela Delta

# Histórico completo de operações
print("=" * 60)
print("HISTÓRICO DE VERSÕES — CAMADA SILVER")
print("=" * 60)

delta_silver.history().select(
    "version",                                                       # número da versão
    "timestamp",                                                     # quando foi executado
    "operation",                                                     # tipo de operação
    "operationParameters"                                            # parâmetros da operação
).show(truncate=50)

HISTÓRICO DE VERSÕES — CAMADA SILVER
+-------+-------------------+---------+--------------------------------------------------+
|version|          timestamp|operation|                               operationParameters|
+-------+-------------------+---------+--------------------------------------------------+
|      6|2026-05-19 23:33:37|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      5|2026-05-19 23:33:33|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      4|2026-05-19 02:16:47|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      3|2026-05-19 02:16:43|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      2|2026-05-15 21:35:30|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      1|2026-05-15 21:33:39|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
|      0|2026-05-15 21:15:59|    WRITE|{mode -> Overwrite, statsOnLoad -> false, parti...|
+-------+-------------------+---------+--------------

#### Histórico de Versões — Silver

| Versão | Data | Operação |
|--------|------|----------|
| 0 | 15/05 21:15 | Primeira carga — Silver criada |
| 1 | 15/05 21:33 | Reprocessamento |
| 2 | 15/05 21:35 | Reprocessamento |
| 3 | 19/05 02:16 | Reprocessamento |
| 4 | 19/05 02:16 | Reprocessamento |
| 5 | 19/05 23:33 | Pipeline Job — execução automática |
| 6 | 19/05 23:33 | Pipeline Job — execução automática |

#### Insight
Versões 5 e 6 foram geradas pelo Job orquestrado —
confirma que o pipeline está funcionando corretamente!

### Célula 2 — Consultando versão anterior

In [0]:
# Consultando a versão 0 — primeira carga da Silver
print("=" * 60)
print("VERSÃO 0 — PRIMEIRA CARGA DA SILVER")
print("=" * 60)

df_v0 = (spark.read
    .format("delta")
    .option("versionAsOf", 0)                                        # versão específica
    .load(SILVER_PATH)
)

print(f"Linhas na versão 0: {df_v0.count():,}")
print(f"Colunas:            {len(df_v0.columns)}")
print()

# Comparando com versão atual
df_atual = spark.read.format("delta").load(SILVER_PATH)
print(f"Linhas na versão atual: {df_atual.count():,}")
print(f"Colunas:                {len(df_atual.columns)}")
print()

# Diferença
diff = df_atual.count() - df_v0.count()
print(f"Diferença de registros: {diff}")
print(f"Status: {'✅ Igual' if diff == 0 else '⚠️ Diferente'}")

VERSÃO 0 — PRIMEIRA CARGA DA SILVER
Linhas na versão 0: 8,469
Colunas:            18

Linhas na versão atual: 8,469
Colunas:                18

Diferença de registros: 0
Status: ✅ Igual


### Célula 3 — Consultando por timestamp

In [0]:
# Consultando por timestamp — após primeira gravação
print("=" * 60)
print("CONSULTA POR TIMESTAMP — 15/05/2026 21:16")
print("=" * 60)

df_timestamp = (spark.read
    .format("delta")
    .option("timestampAsOf", "2026-05-15 21:16:00")                 # após primeira gravação
    .load(SILVER_PATH)
)

print(f"Linhas em 15/05/2026 21:16: {df_timestamp.count():,}")
print(f"Colunas:                    {len(df_timestamp.columns)}")
print()

# Verificando o _loaded_at dessa versão
print("Amostra do _loaded_at nessa versão:")
df_timestamp.select("Ticket_ID", "_loaded_at").show(3)             # quando foi carregado

CONSULTA POR TIMESTAMP — 15/05/2026 21:16
Linhas em 15/05/2026 21:16: 8,469
Colunas:                    18

Amostra do _loaded_at nessa versão:
+---------+--------------------+
|Ticket_ID|          _loaded_at|
+---------+--------------------+
|       12|2026-05-15 21:15:...|
|       18|2026-05-15 21:15:...|
|       38|2026-05-15 21:15:...|
+---------+--------------------+
only showing top 3 rows


### Célula 4 — Simulando erro e restaurando versão anterior

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

print("=" * 60)
print("SIMULAÇÃO — ERRO E RESTAURAÇÃO")
print("=" * 60)

# Versão atual antes do erro
versao_atual = delta_silver.history().select("version").first()[0]
print(f"Versão atual antes do erro: {versao_atual}")

# Simulando um erro — corrompendo dados intencionalmente
print("\n⚠️ Simulando erro — apagando coluna Customer_Age...")
df_corrompido = (spark.read.format("delta").load(SILVER_PATH)
    .drop("Customer_Age")                                            # remove coluna — simula erro
)

df_corrompido.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PATH)

print(f"Colunas após erro: {len(df_corrompido.columns)}")           # deve ser 17 — falta Customer_Age
print("🔴 Dado corrompido!")

# Restaurando para versão anterior
print(f"\n🔄 Restaurando para versão {versao_atual}...")
delta_silver.restoreToVersion(versao_atual)                          # restaura versão anterior

# Validando restauração
df_restaurado = spark.read.format("delta").load(SILVER_PATH)
print(f"Colunas após restauração: {len(df_restaurado.columns)}")    # deve ser 18 — Customer_Age de volta
print(f"Linhas após restauração:  {df_restaurado.count():,}")

print()
if len(df_restaurado.columns) == 18:
    print("✅ Dados restaurados com sucesso!")
else:
    print("🔴 Erro na restauração!")

SIMULAÇÃO — ERRO E RESTAURAÇÃO
Versão atual antes do erro: 6

⚠️ Simulando erro — apagando coluna Customer_Age...
Colunas após erro: 17
🔴 Dado corrompido!

🔄 Restaurando para versão 6...
Colunas após restauração: 18
Linhas após restauração:  8,469

✅ Dados restaurados com sucesso!


#### Simulação de Erro e Restauração — Time Travel

#### Cenário simulado
1. Dado corrompido intencionalmente — coluna `Customer_Age` removida
2. Silver ficou com 17 colunas ao invés de 18
3. Restauração para versão anterior em segundos

#### Resultado
- Tempo de recuperação: **< 5 segundos**
- Linhas recuperadas: **8.469 — sem perda**
- Colunas recuperadas: **18 — completo**

#### Por que isso é importante
Em produção, um erro de ETL pode corromper milhões de registros.
Com Delta Lake Time Travel, a recuperação é instantânea —
sem necessidade de reprocessar toda a pipeline.

### Célula 5 — Histórico final após restauração

In [0]:
print("=" * 60)
print("HISTÓRICO FINAL — APÓS RESTAURAÇÃO")
print("=" * 60)

delta_silver.history().select(
    "version",
    "timestamp",
    "operation"
).show(10)

HISTÓRICO FINAL — APÓS RESTAURAÇÃO
+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      8|2026-05-19 23:52:14|  RESTORE|
|      7|2026-05-19 23:52:03|    WRITE|
|      6|2026-05-19 23:33:37|    WRITE|
|      5|2026-05-19 23:33:33|    WRITE|
|      4|2026-05-19 02:16:47|    WRITE|
|      3|2026-05-19 02:16:43|    WRITE|
|      2|2026-05-15 21:35:30|    WRITE|
|      1|2026-05-15 21:33:39|    WRITE|
|      0|2026-05-15 21:15:59|    WRITE|
+-------+-------------------+---------+



#### Histórico Final — Silver com 9 versões

| Versão | Data | Operação | Descrição |
|--------|------|----------|-----------|
| 0 | 15/05 21:15 | WRITE | Primeira carga da Silver |
| 1 | 15/05 21:33 | WRITE | Reprocessamento |
| 2 | 15/05 21:35 | WRITE | Reprocessamento |
| 3 | 19/05 02:16 | WRITE | Reprocessamento |
| 4 | 19/05 02:16 | WRITE | Reprocessamento |
| 5 | 19/05 23:33 | WRITE | Pipeline Job — automático |
| 6 | 19/05 23:33 | WRITE | Pipeline Job — automático |
| 7 | 19/05 23:52 | WRITE | Simulação de erro 🔴 |
| 8 | 19/05 23:52 | RESTORE | Restauração bem-sucedida ✅ |

#### Conclusão
O Delta Lake registra **auditoria completa** de todas as operações —
quem fez, quando fez e o que foi feito. Em caso de erro,
a recuperação é instantânea sem perda de dados.